In [10]:
import awswrangler as wr
import pandas as pd
import requests
import os
import boto3
import requests
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()



True

In [11]:
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION")
)

In [12]:

# importing s3 bucket locations
raw_s3_bucket = "chigozieobasi"
raw_path_dir = "randomuser_api_airflow_test_not_from_docker2"
csv_path = "personaldata"
path = f"s3://{raw_s3_bucket}/{raw_path_dir}/{csv_path}"


api_url = "https://randomuser.me/api/?results=10"


In [13]:

def extract_api_data():
    """
    fundtion to get data from the API
    Arg: Link to the API
    """
    api_content = requests.get("https://randomuser.me/api/?results=10")
    if api_content.status_code == 200:
        data = api_content.json()
    else:
        print("Error fetching data from API")
        return None
    return data




In [14]:
def transform_male_df(data):
    #data = extract_api_data()
    male = []
    for profile in data["results"]:
        if profile["gender"] == "male":
            male.append(profile["gender"])

    male = pd.DataFrame(male, columns=["male_gender"])
    return male

In [16]:
def load_male_to_s3(df):
    #df = transform_male_df()
    load_dotenv()
    
    session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION"))
    
    raw_s3_bucket = "chigozieobasi"
    raw_path_dir = "randomuser_api_airflow_from_docker2"
    csv_path = "personaldata"
    path = f"s3://{raw_s3_bucket}/{raw_path_dir}/{csv_path}"

    wr.s3.to_parquet(df=df, path=path + "/male", dataset=True, mode="append",boto3_session=session)

**Execute ETL**

In [17]:
raw_data = extract_api_data()
transform_male_data = transform_male_df(raw_data)
load_male_data_to_s3 = load_male_to_s3(transform_male_data)

In [ ]:
def extract_female_df(url):
    data = get_api_data(url)
    female = []
    for profile in data["results"]:
        if profile["gender"] == "female":
            female.append(profile["gender"])

    female = pd.DataFrame(female, columns=["male_gender"])
    return female

In [ ]:
def write_female_to_s3(api_url, path):
    df = extract_female_df(api_url)
    return wr.s3.to_parquet(df=df, path=path + "/female", dataset=True, mode="append")

In [ ]:
def dob_date_df(url):
    data = get_api_data(url)
    dob_date = []
    for profile in data["results"]:
        dob_date.append(profile["dob"]["date"])

    dob = pd.DataFrame(dob_date, columns=["date_of_birth"])

    return dob

In [ ]:
def dob_date_to_s3(api_url, path):
    df = dob_date_df(api_url)
    return wr.s3.to_parquet(df=df, path=path + "/dob", dataset=True, mode="append")

In [ ]:
def full_names_df(url):
    data = get_api_data(url)
    full_names = []
    for profile in data["results"]:
        full_names.append(profile["name"]["first"] + " " + profile["name"]["last"])
    name = pd.DataFrame(full_names, columns=["full_name"])

    return name

In [ ]:
def full_names_to_s3(api_url, path):
    df = full_names_df(api_url)
    return wr.s3.to_parquet(df=df, path=path + "/fullName", dataset=True, mode="append")
